# SMART Rollout Simulation (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/wosac-sim-agents-experiments/blob/main/experiments/smart-constrained/notebooks/smart_rollout_simulation_colab.ipynb)

This notebook is the **simulation stage** only:
- load baseline/variant checkpoints,
- optionally run SMART validation commands,
- write per-model simulation manifests for strict downstream provenance checks.


In [ ]:
# Step 1: Sync repo + deterministic Colab bootstrap
import os
import subprocess
import sys

REPO_URL = "https://github.com/achyutmorang/wosac-sim-agents-experiments.git"
REPO_DIR = "/content/wosac-sim-agents-experiments"

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", "main", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
for path in (REPO_DIR, os.path.join(REPO_DIR, "src")):
    if path not in sys.path:
        sys.path.insert(0, path)

from src.platform import bootstrap_colab_runtime_with_config, wosac_colab_runtime_config
runtime_cfg = wosac_colab_runtime_config(repo_url=REPO_URL, repo_branch="main", repo_dir=REPO_DIR)
bootstrap = bootstrap_colab_runtime_with_config(runtime_cfg)
print("repo_rev:", bootstrap.repo_sync.repo_rev)

if bootstrap.setup.result.get("restart_required", False):
    raise RuntimeError("Runtime restart required. Restart runtime and rerun this cell.")


In [ ]:
# Step 2: Load smart-constrained config + simulation contract context
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

from src.workflows import bootstrap_experiment_pack, load_experiment_config

EXPERIMENT_SLUG = "smart-constrained"
bundle = bootstrap_experiment_pack(slug=EXPERIMENT_SLUG, repo_root=".")
cfg = load_experiment_config(slug=EXPERIMENT_SLUG, repo_root=".")
display(bundle.summary_table)

RUN = dict(cfg.get("run", {}))
SMART = dict(cfg.get("smart", {}))

RUN_NAME = str(RUN.get("run_name", "dev"))
RUN_PREFIX = str(RUN.get("run_prefix", "smart_constrained"))
PERSIST_ROOT = str(RUN.get("persist_root", "/content/drive/MyDrive/wosac_experiments"))
RUN_TAG = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
cfg_hash = hashlib.sha256(json.dumps(cfg, sort_keys=True).encode("utf-8")).hexdigest()
persist_run_dir = Path(PERSIST_ROOT) / f"{RUN_PREFIX}_{RUN_NAME}"
persist_run_dir.mkdir(parents=True, exist_ok=True)

SIM_MANIFESTS_DIR = Path(
    os.environ.get("WOSAC_SIM_MANIFESTS_DIR", str(persist_run_dir / "outputs" / "simulation_manifests"))
).expanduser()
SCENARIO_SET_ID = os.environ.get("WOSAC_SCENARIO_SET_ID", "womd_validation").strip()
SCENARIO_SET_HASH = os.environ.get("WOSAC_SCENARIO_SET_HASH", "").strip()
EVALUATOR_ID = os.environ.get("WOSAC_EVALUATOR_ID", "waymo_open_dataset.sim_agents_metrics.challenge_2025").strip()
METRICS_CONFIG_HASH = os.environ.get("WOSAC_METRICS_CONFIG_HASH", "").strip()
SIM_SEED = int(os.environ.get("WOSAC_SIM_SEED", "2"))

print("RUN_TAG:", RUN_TAG)
print("persist_run_dir:", persist_run_dir)
print("SIM_MANIFESTS_DIR:", SIM_MANIFESTS_DIR)
print("SCENARIO_SET_ID:", SCENARIO_SET_ID)
print("SCENARIO_SET_HASH:", SCENARIO_SET_HASH or "<set this for strict provenance>")
print("EVALUATOR_ID:", EVALUATOR_ID)
print("METRICS_CONFIG_HASH:", METRICS_CONFIG_HASH or "<set this for strict provenance>")
print("cfg_hash:", cfg_hash)


In [ ]:
# Step 3: Define checkpoints to simulate
# Required env vars:
#   SMART_BASELINE_CKPT=/content/drive/.../smart_baseline.ckpt
# Optional:
#   SMART_VARIANT_CKPTS=/content/drive/.../variant1.ckpt,/content/drive/.../variant2.ckpt

baseline_ckpt = os.environ.get("SMART_BASELINE_CKPT", "").strip()
variant_ckpts = [v.strip() for v in os.environ.get("SMART_VARIANT_CKPTS", "").split(",") if v.strip()]

assert baseline_ckpt, "Set SMART_BASELINE_CKPT before running simulation notebook"

models = [{"model_id": "smart_baseline", "checkpoint_path": baseline_ckpt, "env": {}}]
for idx, ckpt in enumerate(variant_ckpts, start=1):
    models.append({"model_id": f"variant_{idx}", "checkpoint_path": ckpt, "env": {}})

print("num_models:", len(models))
for m in models:
    print(m["model_id"], "->", m["checkpoint_path"])


In [ ]:
# Step 4: Build simulation command plan (no metric ranking in this notebook)
from src.workflows import run_smart_eval_flow

flow_bundle = run_smart_eval_flow(
    run_tag=RUN_TAG,
    run_name=RUN_NAME,
    run_prefix=RUN_PREFIX,
    persist_root=PERSIST_ROOT,
    repo_root=".",
    smart_repo_url=str(SMART.get("repo_url", "https://github.com/rainmaker22/SMART.git")),
    smart_repo_branch=str(SMART.get("branch", "main")),
    smart_repo_dir=str(SMART.get("repo_dir", "/content/SMART")),
    smart_val_config=str(SMART.get("val_config", "configs/validation/validation_scalable.yaml")),
    sync_smart_repo=True,
    models=models,
    strict_contract=False,
    require_metrics_binding=False,
    verify_checkpoint_hash=False,
)

print("summary:")
print(json.dumps(flow_bundle.summary, indent=2, sort_keys=True))
print("first validate command:")
print(flow_bundle.models[0]["validate_cmd"])


In [ ]:
# Step 5: Optional simulation execution (closed-loop rollout generation stage)
RUN_SIM_ALL = False
MODEL_IDS_TO_RUN = []  # e.g. ["smart_baseline", "variant_1"]

if RUN_SIM_ALL:
    selected = list(flow_bundle.models)
elif MODEL_IDS_TO_RUN:
    selected = [m for m in flow_bundle.models if m["model_id"] in set(MODEL_IDS_TO_RUN)]
else:
    selected = []

for idx, model in enumerate(selected, start=1):
    cmd = model["validate_cmd"]
    print(f"[simulate {idx}/{len(selected)}] {model['model_id']} -> {cmd}")
    subprocess.run(["bash", "-lc", cmd], check=True)

print("Simulation execution block done.")


In [ ]:
# Step 6: Write per-model simulation manifests + run artifact
from src.workflows import sha256_file, write_simulation_manifest

SIM_MANIFESTS_DIR.mkdir(parents=True, exist_ok=True)
manifest_rows = []

for model in flow_bundle.models:
    model_id = str(model.get("model_id", "")).strip()
    ckpt_path = str(model.get("checkpoint_path", "")).strip()
    manifest_path = SIM_MANIFESTS_DIR / f"{model_id}_simulation_manifest.json"
    manifest = write_simulation_manifest(
        manifest_path,
        {
            "created_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
            "run_tag": RUN_TAG,
            "run_name": RUN_NAME,
            "run_prefix": RUN_PREFIX,
            "model_id": model_id,
            "checkpoint_path": ckpt_path,
            "checkpoint_sha256": sha256_file(ckpt_path),
            "validate_cmd": str(model.get("validate_cmd", "")),
            "scenario_set_id": SCENARIO_SET_ID,
            "scenario_set_hash": SCENARIO_SET_HASH,
            "evaluator_id": EVALUATOR_ID,
            "metrics_config_hash": METRICS_CONFIG_HASH,
            "n_rollouts": int(RUN.get("n_rollouts_per_scenario", 32)),
            "num_history_seconds": int(RUN.get("num_history_seconds", 1)),
            "num_future_seconds": int(RUN.get("num_future_seconds", 8)),
            "seed": int(SIM_SEED),
            "smart_repo_rev": str((flow_bundle.summary.get("smart_repo_sync", {}) or {}).get("repo_rev", "unknown")),
            "source_notebook": "smart_rollout_simulation_colab.ipynb",
        },
    )
    manifest_rows.append({
        "model_id": model_id,
        "manifest_json": str(manifest_path),
        "manifest_sha256": manifest.get("manifest_sha256", ""),
        "checkpoint_sha256": manifest.get("checkpoint_sha256", ""),
    })

payload = {
    "run_id": "smart_rollout_simulation_v0",
    "run_tag": RUN_TAG,
    "status": str(flow_bundle.summary.get("status", "unknown")),
    "simulation_manifests_dir": str(SIM_MANIFESTS_DIR),
    "models": manifest_rows,
    "flow_summary": flow_bundle.summary,
    "provenance": {
        "config_hash": cfg_hash,
        "created_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "experiment_slug": EXPERIMENT_SLUG,
    },
}

repo_path = Path("experiments/smart-constrained/results/smart_rollout_simulation_v0.json")
repo_path.parent.mkdir(parents=True, exist_ok=True)
repo_path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")

drive_path = persist_run_dir / "outputs" / "smart_rollout_simulation_v0.json"
drive_path.parent.mkdir(parents=True, exist_ok=True)
drive_path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")

print("repo_path:", repo_path)
print("drive_path:", drive_path)
print("num_manifests:", len(manifest_rows))
